In [ ]:
import torch
from functools import wraps

# 1. Capture the original, uncorrupted load function
_original_torch_load = torch.load

# 2. Create a wrapper that forces weights_only=False
@wraps(_original_torch_load)
def _patched_torch_load(*args, **kwargs):
    # This specifically fixes the PyTorch 2.6 security restriction
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

# 3. Replace the global load function with our safe version
torch.load = _patched_torch_load

print(f"PyTorch {torch.__version__} fixed. Dataset loading is now enabled.")

In [ ]:
!pip install torch torch-geometric pygod scikit-learn==1.8 pandas matplotlib seaborn networkx -q

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile, os

with zipfile.ZipFile(list(uploaded.keys())[0]) as z:
    z.extractall('project')

os.chdir('project')
os.listdir('.')  # should show run_real.py, data/, detectors/, utils/, etc.

In [ ]:
!nvidia-smi
!python --version

import torch, torch_geometric, pygod, sklearn, numpy as np, pandas as pd

print(f"PyTorch:        {torch.__version__}")
print(f"PyG:            {torch_geometric.__version__}")
print(f"PyGOD:          {pygod.__version__}")
print(f"scikit-learn:   {sklearn.__version__}")
print(f"NumPy:          {np.__version__}")
print(f"pandas:         {pd.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import sys
sys.path.insert(0, '.')
from data.real import load_all

datasets = load_all()
print(f"\n{'Dataset':<12} {'Nodes':>8} {'Edges':>10} {'Features':>10} {'Anomalies':>10} {'Ratio':>8}")
print("-" * 55)
for name, data in datasets.items():
    n_anom = int(data.y.sum())
    print(f"{name:<12} {data.num_nodes:>8} {data.num_edges:>10} "
          f"{data.x.shape[1]:>10} {n_anom:>10} {n_anom/data.num_nodes:>7.1%}")

In [ ]:
%env REAL_EPOCHS=50
!python run_real.py